# GRIB: modifying metadata

This notebook demonstrates how to modify the metadata in GRIB fields.

First we read some GRIB data containing pressure level fields.

In [1]:
import datetime

import earthkit.data as ekd

fl = ekd.from_source("sample", "tuv_pl.grib").to_fieldlist()

tuv_pl.grib:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

We will use the first field in the rest of the notebook.

In [2]:
f = fl[0]
f.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll


## Using set()

In [3]:
f1 = f.set({"parameter.variable": "u", "parameter.units": "m/s", "vertical.level": 500})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll


In [4]:
print(f.get("metadata.shortName"))
try:
    f.set({"metadata.shortName": "u"})
except Exception as e:
    print(e)

t
'Key metadata.shortName cannot be set on the field.'


## Setting time

Setting keys for the time field component allows using multiple formats. By default a "datetime" key takes a datatime.datetime object and a "step" key takes a datatime.timedelta object.

In [5]:
f1 = f.set({"time.base_datetime": datetime.datetime(2000, 12, 18, 12), "time.step": datetime.timedelta(hours=6)})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2000-12-18 18:00:00,2000-12-18 12:00:00,0 days 06:00:00,1000,pressure,0,regular_ll


On top of that, we can also use many compatible formats, e.g:
  - for datetime: ISO date strings, numpy datetime64 values, integers as yyyymmdd (the hour is assumed to be 0 in this case)
  - for timedelta: integers (as hours), strings like "6s", "6m", "6h" (for seconds, minutes or hours)

In [6]:
f1 = f.set({"time.base_datetime": "2000-12-18T12", "time.step": 6})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2000-12-18 18:00:00,2000-12-18 12:00:00,0 days 06:00:00,1000,pressure,0,regular_ll


Setting the step will automatically update the a valid time too.

In [7]:
f1 = f.set({"time.step": "10s"})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:10,2018-08-01 12:00:00,0 days 00:00:10,1000,pressure,0,regular_ll


## Setting components

In [8]:
f1 = f.set(time={"base_datetime": "2000-12-18T12", "step": 6})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2000-12-18 18:00:00,2000-12-18 12:00:00,0 days 06:00:00,1000,pressure,0,regular_ll


If the dict is not fully specifying the component an exception is raised. E.g. "step" on it is own does not define a time component.

In [9]:
try:
    f.set(time={"step": 6})
except Exception as e:
    print(e)

Cannot create ForecastTime from keys: ['step'].


## Saving the modified field to disk

We change the level and save the modified field into a GRIB file.

In [10]:
f1 = f.set({"vertical.level": 500})
f1.to_target("file", "_res_lev.grib")

# read back the data and compare the values in the first field
f1_w = ekd.from_source("file", "_res_lev.grib").to_fieldlist()
f1_w.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,500,pressure,0,regular_ll


## Changing raw GRIB metadata

In [11]:
encoder = ekd.create_encoder("grib")
r = encoder.encode(template=f, metadata={"shortName": "u"})
f1 = r.to_field()
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
